In [1]:
import os
import re
import numpy as np
import rasterio
import geopandas as gpd
import xarray as xr
from rasterio.mask import mask
from datetime import datetime

In [2]:


def read_tif_files(directory, pattern="SWE"):
    tif_files = [os.path.join(directory, f) for f in os.listdir(directory) if f.endswith('.tif')]
    # Update to pattern check if "SWE" in filename
    tif_files = [f for f in tif_files if pattern in f]
    return tif_files

def extract_date_from_filename(filename):
    date_str = re.search(r'\d{8}', filename).group()
    return datetime.strptime(date_str, '%Y%m%d')

def reproject_geometries(hrus, src_crs):
    hrus_reprojected = hrus.to_crs(src_crs)
    return hrus_reprojected

def calculate_mean_swe(tif_file, hrus):
    """
    Calculate mean snow water equivalent (in meters) for each HRU polygon in tif_file.
    Assumes the tif file contains SWE in meters; nodata value is -9999.
    Screens out any SWE values below zero.
    """
    with rasterio.open(tif_file) as src:
        src_crs = src.crs.to_string()
        hrus_reprojected = reproject_geometries(hrus, src_crs)

        mean_swe = []
        for hru in hrus_reprojected.itertuples():
            hru_mask = [hru.geometry]
            # NOTE: src.nodata is typically set properly, but we explicitly specify nodata because file spec is known
            swe_hru, _ = mask(src, hru_mask, crop=True, filled=True, nodata=-9999)
            swe_hru = np.ma.masked_equal(swe_hru, -9999)
            # Mask out negative values as well
            swe_hru = np.ma.masked_less(swe_hru, 0)
            # Data is in meters, NO unit conversion.
            mean_swe_m = swe_hru.mean()
            mean_swe.append((hru.HRU_ID, mean_swe_m))

    return mean_swe

def aggregate_swe(tif_files, hrus):
    swe = {}
    for tif_file in tif_files:
        date = extract_date_from_filename(tif_file)
        mean_swe = calculate_mean_swe(tif_file, hrus)
        for hru_id, mean_swe_value in mean_swe:
            if hru_id not in swe:
                swe[hru_id] = []
            swe[hru_id].append((date, mean_swe_value))
    return swe

def write_to_netcdf_cf_conventions(swe_data, output_file):
    """
    Write the aggregated SWE data to netCDF4 file with CF-conventions.
    Variable output is snow water equivalent in meters.
    """
    hru_ids = list(map(str, swe_data.keys()))
    dates = sorted({date for values in swe_data.values() for date, _ in values})
    dates_np = np.array(dates)
    # Convert to numpy datetime64 for xarray CF compliance with time axis
    np_time = np.array([np.datetime64(d) for d in dates_np])

    swe_output = np.full((len(hru_ids), len(dates)), np.nan)

    for i, hru_id in enumerate(hru_ids):
        for date, mean_swe in swe_data[int(hru_id)]:
            j = dates.index(date)
            swe_output[i, j] = mean_swe

    # CF Conventions for SWE: variable name 'swe', units 'm', 2D (hru, time)
    ds = xr.Dataset(
        {
            "swe": (
                ["hru", "time"],
                swe_output,
                {
                    "long_name": "Snow Water Equivalent",
                    "standard_name": "lwe_thickness_of_surface_snow_amount",
                    "units": "m",
                    "coordinates": "hru time",
                    "missing_value": np.nan,
                    "description": "Mean snow water equivalent in meters for each HRU polygon"
                }
            ),
        },
        coords={
            "HRU_ID": ("hru", hru_ids, {"long_name": "Hydrologic Response Unit ID"}),
            "time": ("time", np_time, {"long_name": "Time", "standard_name": "time"})
        },
        attrs={
            "title": "Snow Water Equivalent aggregated by HRU",
            "summary": "Mean SWE over HRUs from Lidar data, following CF conventions",
            "Conventions": "CF-1.8",
            "institution": "Your Institution Here",
            "history": f"Created {datetime.utcnow().isoformat()} UTC",
            "source": "Lidar derived snow water equivalent",
        }
    )

    # Set encoding for CF-compliance, using NaN as _FillValue since nodata is -9999 in input but arrays use np.nan for missing
    ds['swe'].encoding.update({"zlib": True, "complevel": 4, "_FillValue": np.nan})
    ds.to_netcdf(output_file, format="NETCDF4", encoding={"swe": {"_FillValue": np.nan}})


In [3]:

def main():
    # Paths for input/output files
    directory = '/anvil/projects/x-ees240082/users/dcasson/lidar/tuolumne'
    hrus_file = '/anvil/projects/x-ees240082/users/dcasson/gpep/tuolumne/gis/tuolumne_tdx.gpkg'
    swe_netcdf_output = '/anvil/projects/x-ees240082/users/dcasson/lidar/tuolumne_lidar_swe_data.nc'

    # Read HRU polygons
    hrus = gpd.read_file(hrus_file)

    # Find all SWE tiffs (for Lidar processed output)
    swe_tif_files = read_tif_files(directory, pattern='SWE')
    if not swe_tif_files:
        print(f"No SWE tif files found in {directory}")
        return

    # Aggregate SWE for each HRU
    swe_data = aggregate_swe(swe_tif_files, hrus)

    # Write out to NetCDF with CF conventions
    write_to_netcdf_cf_conventions(swe_data, swe_netcdf_output)
    print(f"Wrote SWE HRU-aggregated data to {swe_netcdf_output}")

if __name__ == "__main__":
    main()

/tmp/ipykernel_1731089/1651731524.py:93: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "history": f"Created {datetime.utcnow().isoformat()} UTC",


Wrote SWE HRU-aggregated data to /anvil/projects/x-ees240082/users/dcasson/lidar/tuolumne_lidar_swe_data.nc
